In [118]:
import torch
from huggingface_hub import login
from datasets import load_dataset
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig

from torch.utils.data import Dataset, DataLoader

In [110]:
login()

In [111]:
STRESS_SUPPORT_STATEMENT = "It sounds like you are carrying a lot of stress and feeling completely exhausted. Please remember to be gentle with yourself today, and consider stepping away for a short break to rest."

GENERAL_SUPPORT_STATEMENT = "Thank you for sharing your thoughts today. I hope things continue to go smoothly for you and that you have a wonderful, peaceful day ahead!"

BASE_INSTRUCTION = "Analyze this Reddit post for signs of psychological stress. Provide a brief, supportive response in 1–2 sentences. If stress is present, acknowledge feelings and suggest one helpful action. If no stress, provide encouragement. Be empathetic but avoid medical advice. Post: "

In [112]:
class Dreaddit(Dataset):
    def __init__(self, texts, text_tokenizer):
        self.texts = texts
        self.tokenizer = text_tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = self.texts[idx]

        prompt_len = len(item["prompt"])
        tokenizer_data = self.tokenizer(item).data

        # Raw token values
        input_ids = tokenizer_data["input_ids"]

        #Determines what a real token is
        attn_mask = tokenizer_data["attention_mask"]

        # Determines what gets scored
        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        return input_ids, attn_mask, labels

In [113]:
def build_features(sample):
    stress = sample["label"]
    response = STRESS_SUPPORT_STATEMENT if stress == 1 else GENERAL_SUPPORT_STATEMENT
    prompt = f"instruction:{BASE_INSTRUCTION} post:{sample["text"]} response: "

    return {"prompt": prompt, "response": response}

In [114]:
ds = load_dataset("hutchii/dreaddit")

In [71]:
train = ds["train"]
prompts = train.map(build_features).select_columns("prompt")
prompt_data = prompts.data["prompt"].to_pylist()

In [115]:
print(len(prompt_data))
print(type(prompt_data))

2838
<class 'list'>


In [116]:
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM3-3B")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)
dreaddit = Dreaddit(texts=prompt_data, text_tokenizer=tokenizer)
dl = DataLoader(dataset=dreaddit, batch_size=32, collate_fn=collate_fn)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, peft_config=lora_config)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [ ]:
MAX_EPOCH = 5
optimizer = torch.optim.Adam(peft_model.parameters(), lr=.01)
loss_fn = CrossEntropyLoss()

for e in range(MAX_EPOCH):
    for x,y in dl:
        peft_model.train()
        resp_tokens = peft_model(x)
        loss = loss_fn(resp_tokens, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
